# Lab-3 Turning functions into tools

In [ ]:
import json
from dotenv import load_dotenv
_ = load_dotenv()

In [48]:
from google import genai
from google.genai import types
import os

CLIENT = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

### Build your first tool

In [4]:
from datetime import datetime

def get_current_time():
    """
    REturns the current time as a string.
    """
    return datetime.now().strftime("%H:%M:%S")

In [5]:
# Test function
get_current_time()

'20:51:09'

### Turning the function into an LLM tool

In [ ]:
prompt = "What time is it?"
config = types.GenerateContentConfig(tools=[get_current_time])
response = CLIENT.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
    config=config,
)
print(response.text)

The current time is 21:04:05.


In [19]:
# print all available attributes and methods
# print(dir(response))

# see the whole response
from pprint import pprint
pprint(response.to_json_dict())

{'automatic_function_calling_history': [{'parts': [{'text': 'What time is '
                                                            'it?'}],
                                         'role': 'user'},
                                        {'parts': [{'function_call': {'args': {},
                                                                      'name': 'get_current_time'}}],
                                         'role': 'model'},
                                        {'parts': [{'function_response': {'name': 'get_current_time',
                                                                          'response': {'result': '21:04:05'}}}],
                                         'role': 'user'}],
 'candidates': [{'content': {'parts': [{'text': 'The current time is '
                                                '21:04:05.'}],
                             'role': 'model'},
                 'finish_reason': 'STOP',
                 'index': 0}],
 'model_version': 'gemini-2

### Manually defining tools

In [27]:
# Define the function declaration for the model (gemini)
get_current_time_function = {
    "name": "get_current_time",
    "description": "Get the current system time.",
    "parameters": {},
}
tools = types.Tool(function_declarations=[get_current_time_function])
config = types.GenerateContentConfig(tools=[tools])

# Define user prompt
user_prompt = [
    types.Content(
        role="user",
        parts=[types.Part(text="What's the current time?")]
    )
]

In [ ]:
response = CLIENT.models.generate_content(
    model="gemini-2.5-flash",
    contents=user_prompt,
    config=config,
)

In [31]:
import json

pprint(response.to_json_dict())
print(response.candidates[0].content.parts[0].function_call)
print(json.dumps(response.model_dump(), indent=2, default=str))

{'automatic_function_calling_history': [],
 'candidates': [{'content': {'parts': [{'function_call': {'args': {},
                                                          'name': 'get_current_time'}}],
                             'role': 'model'},
                 'finish_reason': 'STOP',
                 'index': 0}],
 'model_version': 'gemini-2.5-flash',
 'usage_metadata': {'candidates_token_count': 12,
                    'prompt_token_count': 37,
                    'total_token_count': 89}}
id=None args={} name='get_current_time'
{
  "candidates": [
    {
      "content": {
        "parts": [
          {
            "video_metadata": null,
            "thought": null,
            "code_execution_result": null,
            "executable_code": null,
            "file_data": null,
            "function_call": {
              "id": null,
              "args": {},
              "name": "get_current_time"
            },
            "function_response": null,
            "inline_data": n

#### Run the function call
if tool call name matches

In [32]:
tool_call = response.candidates[0].content.parts[0].function_call
if tool_call.name == "get_current_time":
    result = get_current_time(**tool_call.args)
    print(f"Function execution result: {result}")

Function execution result: 21:38:07


### Give the LLM more tools
3 new tools

In [80]:
import requests
import qrcode
from qrcode.image.styledpil import StyledPilImage

def get_weather_from_ip():
    """
    Gets the current, high, and low temperature in Celsius for the user's location and returns it to the user.
    """
    lat, lon = requests.get('https://ipinfo.io/json').json()['loc'].split(',')
    
    # Set parameters for the weather API call
    params = {
        "latitude": lat,
        "longitude": lon,
        "current": "temperature_2m",
        "daily": "temperature_2m_max,temperature_2m_min",
        "temperature_unit": "fahrenheit",
        "timezone": "auto"
    }

     # Get weather data
    weather_data = requests.get("https://api.open-meteo.com/v1/forecast", params=params).json()

    def convert_f_to_c(temp_in_f: float) -> float:
        temp_c = (temp_in_f - 32) * (5/9)
        return temp_c
    
    current_c = convert_f_to_c(weather_data['current']['temperature_2m'])
    high_c = convert_f_to_c(weather_data['daily']['temperature_2m_max'][0])
    low_c = convert_f_to_c(weather_data['daily']['temperature_2m_min'][0])

    return (
        f"Current: {current_c:.2f} °C, "
        f"High: {high_c:.2f} °C,"
        f"Low: {low_c:.2f} °C"
    )

def write_txt_file(file_path: str, content: str) -> str:
    """
    Write a string into a .txt file (overwrites if exists).
    Args:
        file_path (str): Destination path.
        content (str): Text to write.
    Returns:
        str: Path to the written file.
    """
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)
    return file_path

def generate_qr_code(data: str, filename: str, image_path: str) -> str:
    """Generate a QR code image given data and an image path.

    Args:
        data: Text or URL to encode
        filename: Name for the output PNG file (without extension)
        image_path: Path to the image to be used in the QR code
    """
    qr = qrcode.QRCode(error_correction=qrcode.constants.ERROR_CORRECT_H)
    qr.add_data(data)

    img = qr.make_image(image_factory=StyledPilImage, embedded_image_path=image_path)
    output_file = f"{filename}.png"
    img.save(output_file)

    return f"QR code saved as {output_file} containing: {data[:50]}..."

In [43]:
# declare functions
get_weather_from_ip_function = {
    "name": "get_weather_from_ip",
    "description": "Gets the current, high, and low temperature in Celsius for the user's location and returns it to the user.",
    "parameters": {},
}

write_txt_file_function = {
    "name": "write_txt_file",
    "description": "Write a string into a .txt file (overwrites if exists).",
    "parameters": {
        "type": "object",
        "properties": {
            "file_path": {
                "type": "string",
                "description": "Destination path.",
            },
            "content": {
                "type": "string",
                "description": "Text to write.",
            },
        },
        "required": ["file_path", "content"]
    },
}

generate_qr_code_function = {
    "name": "generate_qr_code",
    "description": "Generate a QR code image given data and an image path.",
    "parameters": {
        "type": "object",
        "properties": {
            "data": {
                "type": "string",
                "description": "Text or URL to encode",
            },
            "filename": {
                "type": "string",
                "description": "Name for the output PNG file (without extension",
            },
            "image_path": {
                "type": "string",
                "description": "Path to the image to be used in the QR code",
            },
        },
        "required": ["data", "filename", "image_path"]
    },
}

In [ ]:
# Google GenAI Python SDK support automatic function calls
# We can directly pass functions and skip declaration
# and don't need to parse function names and execute on client end

In [85]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

def get_response_from_gemini(contents, config, model = "gemini-2.5-flash"):
    response = CLIENT.models.generate_content(
        model=model,
        contents=contents,
        config=config
    )
    return response.text

In [89]:
toolbox_config = types.GenerateContentConfig(
    tools=[get_current_time, get_weather_from_ip, write_txt_file, generate_qr_code]
)

In [90]:
prompt1 = "Can you get the weather for my location?"
response1 = get_response_from_gemini(prompt1, toolbox_config)
print(response1)

The current temperature is 8.11°C, with a high of 13.17°C and a low of 8.06°C.


In [91]:
prompt2 = "Can you make a txt note for me called reminders.txt that reminds me to call Daniel tomorrow at 7PM?"
response2 = get_response_from_gemini(prompt2, toolbox_config)
print(response2)

I've created a file named "reminders.txt" with the content "Call Daniel tomorrow at 7PM".



In [94]:
prompt3 = "Can you make a QR code for me using my logo that goes to https://thought-bytes.com/? The logo is located at `icon.png`. You can call it test_qr_code."
response3 = get_response_from_gemini(prompt3, toolbox_config)
print(response3)

I have generated a QR code for "https://thought-bytes.com/" using your logo at "icon.png" and saved it as "test_qr_code.png".



#### Using multiple tools

In [96]:
prompt4 = "Can you help me create a qr code that goes to https://thought-bytes.com/ from the image `icon.png`, name the file `test_2_qr? Also write me a txt note with the current weather please."
response4 = get_response_from_gemini(prompt4, toolbox_config)
print(response4)

I have created the QR code and a text file named `weather.txt` containing the current weather information.
